<a href="https://colab.research.google.com/github/HazemmoAlsady/AWN_Graduation_Project/blob/main/01_batch_etl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Raw CSV
   # ↓
 # Cleaning
   # ↓
# Feature Engineering
   # ↓
# Cleaned Parquet

In [4]:
!pip install pyspark --quiet
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pyspark.sql.functions as F

# Spark Session

In [5]:
spark = SparkSession.builder \
    .appName("RetailBatchETL") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("Spark Ready")

Spark Ready


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Load Dataset

In [7]:
file_path = "/content/drive/MyDrive/online_retail_II.csv"

data = spark.read.csv(
    file_path,
    header=True,
    inferSchema=False
)

print(f"Rows: {data.count():,}")
print(f"Columns: {len(data.columns)}")

data.show(5)

Rows: 1,067,371
Columns: 8
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+
|Invoice|StockCode|         Description|Quantity|        InvoiceDate|Price|Customer ID|       Country|
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+
| 489434|    85048|15CM CHRISTMAS GL...|      12|2009-12-01 07:45:00| 6.95|    13085.0|United Kingdom|
| 489434|   79323P|  PINK CHERRY LIGHTS|      12|2009-12-01 07:45:00| 6.75|    13085.0|United Kingdom|
| 489434|   79323W| WHITE CHERRY LIGHTS|      12|2009-12-01 07:45:00| 6.75|    13085.0|United Kingdom|
| 489434|    22041|"RECORD FRAME 7""...|      48|2009-12-01 07:45:00|  2.1|    13085.0|United Kingdom|
| 489434|    21232|STRAWBERRY CERAMI...|      24|2009-12-01 07:45:00| 1.25|    13085.0|United Kingdom|
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+
only showing top 5 rows


# Data Quality Report

In [8]:
data = data.withColumn(
    "Quantity",
    col("Quantity").cast("integer")
)

data = data.withColumn(
    "Price",
    col("Price").cast("double")
)

In [9]:
total_rows = data.count()

null_counts = data.select([
    F.sum(col(c).isNull().cast("int")).alias(c)
    for c in data.columns
]).collect()[0].asDict()

print("="*50)
print("DATA QUALITY REPORT")
print("="*50)

for c, n in null_counts.items():
    print(f"{c:<20} {n:>8,}  {n/total_rows*100:.2f}%")

DATA QUALITY REPORT
Invoice                     0  0.00%
StockCode                   0  0.00%
Description             4,382  0.41%
Quantity                    0  0.00%
InvoiceDate                 0  0.00%
Price                       0  0.00%
Customer ID           243,007  22.77%
Country                     0  0.00%


In [10]:
dup_count = data.count() - data.dropDuplicates().count()

print(f"Duplicate Rows: {dup_count:,}")

Duplicate Rows: 34,335


In [11]:
neg_qty = data.filter(col("Quantity") < 0).count()

print(f"Negative Quantity Rows: {neg_qty:,}")

Negative Quantity Rows: 22,950


# ETL Cleaning

In [12]:
data = data.dropna(
    subset=[
        "Customer ID",
        "Description",
        "Invoice"
    ]
)

In [13]:
data = data.filter(
    ~col("Invoice").startswith("C")
)

In [14]:
data = data.filter(
    (col("Quantity") > 0) &
    (col("Price") > 0)
)

In [15]:
data = data.dropDuplicates()

In [16]:
data = data.withColumn(
    "InvoiceDate",
    col("InvoiceDate").cast("timestamp")
)

In [17]:
data = data.withColumnRenamed(
    "Customer ID",
    "Customer_ID"
)

# Feature Engineering

In [18]:
data = data.withColumn(
    "TotalPrice",
    round(col("Quantity") * col("Price"), 2)
)

In [19]:
data = data.withColumn(
    "Year",
    year(col("InvoiceDate"))
)

data = data.withColumn(
    "Month",
    month(col("InvoiceDate"))
)

data = data.withColumn(
    "Quarter",
    quarter(col("InvoiceDate"))
)

# Cache Dataset

In [20]:
data.cache()

DataFrame[Invoice: string, StockCode: string, Description: string, Quantity: int, InvoiceDate: timestamp, Price: double, Customer_ID: string, Country: string, TotalPrice: double, Year: int, Month: int, Quarter: int]

In [21]:
print(f"Clean Rows: {data.count():,}")

data.printSchema()

data.show(5)

Clean Rows: 779,425
root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- Price: double (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- TotalPrice: double (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Quarter: integer (nullable = true)

+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+----------+----+-----+-------+
|Invoice|StockCode|         Description|Quantity|        InvoiceDate|Price|Customer_ID|       Country|TotalPrice|Year|Month|Quarter|
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+----------+----+-----+-------+
| 489460|    84568|GIRLS ALPHABET IR...|     288|2009-12-01 10:46:00| 0.21|    16167.0|Un

In [22]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
import pandas as pd
data_pd = data.toPandas()

In [24]:
data_pd.to_parquet(
    "/content/cleaned_data.parquet",
    index=False
)